In [2]:
from math import gcd
from math import isqrt

In [3]:
def search_solutions(a, b, c, d, Sj, Sj2, n):
    """
    Returns all solutions (a_i,b_i,c_i,d_i,k_i)
    satisfying the reduced system.
    """

    # Necessary condition
    if a * c * Sj != b * d * Sj2:
        return []

    g = gcd(b, c)
    bp = b // g
    cp = c // g

    rhs = a * b * Sj

    if rhs % (bp * bp) != 0:
        return []

    M = rhs // (bp * bp)

    solutions = []

    def build_solution(ts, ks):

        a_list = []
        b_list = []
        c_list = []
        d_list = []

        for t, k in zip(ts, ks):

            bi = bp * t
            ci = cp * t

            # divisibility constraints
            if (k * bi) % Sj != 0:
                return

            if (k * ci) % Sj2 != 0:
                return

            ai = (k * bi) // Sj
            di = (k * ci) // Sj2

            a_list.append(ai)
            b_list.append(bi)
            c_list.append(ci)
            d_list.append(di)

        # Verify original equations exactly

        if sum(ai*bi for ai,bi in zip(a_list,b_list)) != a*b:
            return

        if sum(ci*di for ci,di in zip(c_list,d_list)) != c*d:
            return

        if sum(ai*ci for ai,ci in zip(a_list,c_list)) != a*c:
            return

        if sum(bi*di for bi,di in zip(b_list,d_list)) != b*d:
            return

        solutions.append({
            "a_i": a_list,
            "b_i": b_list,
            "c_i": c_list,
            "d_i": d_list,
            "k_i": ks,
        })

    def recurse(idx, remaining, ts, ks):

        if idx == n:
            if remaining == 0:
                build_solution(ts, ks)
            return

        terms_left = n - idx

        # Every remaining term contributes at least 1
        if remaining < terms_left:
            return

        # max_t = int(remaining**0.5)
        max_t = isqrt(remaining)

        for t in range(1, max_t + 1):

            t2 = t * t

            max_k = remaining // t2

            for k in range(1, max_k + 1):

                contrib = k * t2

                recurse(
                    idx + 1,
                    remaining - contrib,
                    ts + [t],
                    ks + [k]
                )

    recurse(0, M, [], [])

    return solutions

In [4]:
sols = search_solutions(
    a=3,
    b=2,
    c=2,
    d=3,
    Sj=4,
    Sj2=4,
    n=2
)

print(len(sols))
for s in sols[:10]:
    print(s)

15
{'a_i': [1, 5], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [1, 5], 'k_i': [4, 20]}
{'a_i': [2, 4], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [2, 4], 'k_i': [8, 16]}
{'a_i': [2, 2], 'b_i': [1, 2], 'c_i': [1, 2], 'd_i': [2, 2], 'k_i': [8, 4]}
{'a_i': [2, 1], 'b_i': [1, 4], 'c_i': [1, 4], 'd_i': [2, 1], 'k_i': [8, 1]}
{'a_i': [3, 3], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [3, 3], 'k_i': [12, 12]}
{'a_i': [4, 2], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [4, 2], 'k_i': [16, 8]}
{'a_i': [4, 1], 'b_i': [1, 2], 'c_i': [1, 2], 'd_i': [4, 1], 'k_i': [16, 2]}
{'a_i': [5, 1], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [5, 1], 'k_i': [20, 4]}
{'a_i': [1, 4], 'b_i': [2, 1], 'c_i': [2, 1], 'd_i': [1, 4], 'k_i': [2, 16]}
{'a_i': [1, 2], 'b_i': [2, 2], 'c_i': [2, 2], 'd_i': [1, 2], 'k_i': [2, 4]}


In [5]:
from functools import lru_cache
from math import isqrt

def decomposition_dp(M, n):

    @lru_cache(None)
    def solve(remaining, slots, min_t, min_k):

        if slots == 0:
            return [()] if remaining == 0 else []

        if remaining < slots:
            return []

        results = []

        max_t = isqrt(remaining)

        for t in range(min_t, max_t + 1):

            if t == min_t:
                start_k = min_k
            else:
                start_k = 1

            max_k = remaining // (t*t)

            for k in range(start_k, max_k + 1):

                contribution = k*t*t

                future_min = slots - 1

                if remaining - contribution < future_min:
                    break

                tails = solve(
                    remaining - contribution,
                    slots - 1,
                    t,
                    k
                )

                pair = (t,k)

                for tail in tails:
                    results.append((pair,) + tail)

        return results

    return solve(M, n, 1, 1)

In [6]:
decomposition_dp(24,2)

[((1, 1), (1, 23)),
 ((1, 2), (1, 22)),
 ((1, 3), (1, 21)),
 ((1, 4), (1, 20)),
 ((1, 4), (2, 5)),
 ((1, 5), (1, 19)),
 ((1, 6), (1, 18)),
 ((1, 6), (3, 2)),
 ((1, 7), (1, 17)),
 ((1, 8), (1, 16)),
 ((1, 8), (2, 4)),
 ((1, 8), (4, 1)),
 ((1, 9), (1, 15)),
 ((1, 10), (1, 14)),
 ((1, 11), (1, 13)),
 ((1, 12), (1, 12)),
 ((1, 12), (2, 3)),
 ((1, 15), (3, 1)),
 ((1, 16), (2, 2)),
 ((1, 20), (2, 1)),
 ((2, 1), (2, 5)),
 ((2, 2), (2, 4)),
 ((2, 2), (4, 1)),
 ((2, 3), (2, 3))]

In [7]:
from math import gcd

def recover_solution(solution, a, b, c, d, Sj, Sj2):
    """
    solution = ((t1,k1),...,(tn,kn))
    """

    g = gcd(b, c)
    bp = b // g
    cp = c // g

    a_list = []
    b_list = []
    c_list = []
    d_list = []

    for t, k in solution:

        bi = bp * t
        ci = cp * t

        ai = (k * bi) // Sj
        di = (k * ci) // Sj2

        a_list.append(ai)
        b_list.append(bi)
        c_list.append(ci)
        d_list.append(di)

    if sum(ai*bi for ai,bi in zip(a_list,b_list)) != a*b:
        return

    if sum(ci*di for ci,di in zip(c_list,d_list)) != c*d:
        return

    if sum(ai*ci for ai,ci in zip(a_list,c_list)) != a*c:
        return

    if sum(bi*di for bi,di in zip(b_list,d_list)) != b*d:
        return

    return {
        "a_i": a_list,
        "b_i": b_list,
        "c_i": c_list,
        "d_i": d_list,
        "k_i": [k for t, k in solution]
    }

In [8]:
a, b, c, d = 3, 2, 2, 3
Sj, Sj2 = 4, 4
g = gcd(b,c)
bp = b // g
M = (a*b*Sj) // (bp**2)
n=2

decomps = decomposition_dp(M,n)

solutions = []
for decomp in decomps:
    solution = recover_solution(decomp, a, b, c, d, Sj, Sj2)
    if solution is not None:
        print(solution)
        solutions.append(solution)

{'a_i': [1, 5], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [1, 5], 'k_i': [4, 20]}
{'a_i': [2, 4], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [2, 4], 'k_i': [8, 16]}
{'a_i': [2, 2], 'b_i': [1, 2], 'c_i': [1, 2], 'd_i': [2, 2], 'k_i': [8, 4]}
{'a_i': [2, 1], 'b_i': [1, 4], 'c_i': [1, 4], 'd_i': [2, 1], 'k_i': [8, 1]}
{'a_i': [3, 3], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [3, 3], 'k_i': [12, 12]}
{'a_i': [4, 1], 'b_i': [1, 2], 'c_i': [1, 2], 'd_i': [4, 1], 'k_i': [16, 2]}
{'a_i': [1, 2], 'b_i': [2, 2], 'c_i': [2, 2], 'd_i': [1, 2], 'k_i': [2, 4]}
{'a_i': [1, 1], 'b_i': [2, 4], 'c_i': [2, 4], 'd_i': [1, 1], 'k_i': [2, 1]}


In [15]:
a, b, c, d = 4,3,3,4
Sj, Sj2 = 15,15
g = gcd(b,c)
bp = b // g
M = (a*b*Sj) // (bp**2)
n=2

decomps = decomposition_dp(M,n)

solutions = []
for decomp in decomps:
    solution = recover_solution(decomp, a, b, c, d, Sj, Sj2)
    if solution is not None:
        print(solution)
        solutions.append(solution)

{'a_i': [1, 11], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [1, 11], 'k_i': [15, 165]}
{'a_i': [2, 10], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [2, 10], 'k_i': [30, 150]}
{'a_i': [2, 2], 'b_i': [1, 5], 'c_i': [1, 5], 'd_i': [2, 2], 'k_i': [30, 6]}
{'a_i': [3, 9], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [3, 9], 'k_i': [45, 135]}
{'a_i': [3, 3], 'b_i': [1, 3], 'c_i': [1, 3], 'd_i': [3, 3], 'k_i': [45, 15]}
{'a_i': [4, 8], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [4, 8], 'k_i': [60, 120]}
{'a_i': [4, 4], 'b_i': [1, 2], 'c_i': [1, 2], 'd_i': [4, 4], 'k_i': [60, 30]}
{'a_i': [5, 7], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [5, 7], 'k_i': [75, 105]}
{'a_i': [6, 6], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [6, 6], 'k_i': [90, 90]}
{'a_i': [6, 2], 'b_i': [1, 3], 'c_i': [1, 3], 'd_i': [6, 2], 'k_i': [90, 10]}
{'a_i': [7, 1], 'b_i': [1, 5], 'c_i': [1, 5], 'd_i': [7, 1], 'k_i': [105, 3]}
{'a_i': [8, 2], 'b_i': [1, 2], 'c_i': [1, 2], 'd_i': [8, 2], 'k_i': [120, 15]}
{'a_i': [9, 1], 'b_i': [1, 3], 'c_i': [1, 3], 'd_i': [9

### General Dynamic Programming Algorithm

In [ ]:
from sympy import divisors
import numpy as np

def general_dp_ep(ps, qs, SI, SF, num_paths):
    """Compute set of valid equitable partition unfoldings of a weighted 
    path that maintain the same number of non-returning walks of all lengths.

    ps: array(int)
        List of integers whose i-th entry represents the number of edges from partition i-1 to partition i
    qs: array(int)
        List of integers whose i-th entry represents the number of edges from partition i to partition i-1
    SI: int
        Number of vertices in the initial partition
    SF: int
        Number of vertices in the final partition
    num_paths:  int
        Number of paths to separate the weighted path into
    """
    n = len(ps)
    if n != len(qs):
        print("Length of ps does not match qs")
        return
    
    def _divisors_less_than_m(n, m):
        return [d for d in divisors(int(n)) if d < m]

    ks = [0]*(n+1)
    ks[0] = SI
    ks[-1] = SF
    
    solutions = []

    p_0, q_0 = ps[0], qs[0]
    path_sum = p_0 * q_0
    s_0 = SI
    s_1 = int(SI * p_0 / q_0)
    s_0_factors = _divisors_less_than_m(s_0, s_0)
    s_1_factors = [d for d in divisors()]





ps = [3,2]
qs = [2,3]
SI = 6
SF = 6
num_paths = 2
general_dp_ep(ps, qs, SI, SF, num_paths)

In [24]:
np.array(divisors(42), dtype=int)

array([ 1,  2,  3,  6,  7, 14, 21, 42])

In [32]:
def combinations_leq_m(m, n, minimum=1):
    if n == 0:
        yield ()
        return

    # Need at least n * minimum remaining
    if m < n * minimum:
        return

    max_first = m // n

    for first in range(minimum, max_first + 1):
        for rest in combinations_leq_m(
            m - first,
            n - 1,
            first
        ):
            yield (first,) + rest

In [ ]:
def single_step_ep(p, q, S, num_paths):
    path_total = int(p * q)
    possible_p_combinations = list(combinations_leq_m(path_total, num_paths))

    solutions = []
    
    for P in possible_p_combinations:
        possible_q_combinations = []
        for p_i in P:
            divisor_num = int(S * p_i)
            possible_q_combinations.append([d for d in divisors(divisor_num) if d < S])
        print(P)
        print(possible_q_combinations)

        # Implement branch and bound for optimal searching of viable p_i, q_i
        min_contrib = [0]*(n+1)
        max_contrib = [0]*(n+1)

        for i in range(n-1, -1, -1):
            vals = [P[i]*q for q in possible_q_combinations[i]]

            min_contrib[i] = min_contrib[i+1] + min(vals)
            max_contrib[i] = max_contrib[i+1] + max(vals)

        def recurse(i, current_sum, chosen):

            if i == n:
                if current_sum == path_total:
                    yield tuple(chosen)
                return

            # prune impossible branches
            if current_sum + min_contrib[i] > path_total:
                return

            if current_sum + max_contrib[i] < path_total:
                return

            p = P[i]

            for q in possible_q_combinations[i]:
                yield from recurse(
                    i+1,
                    current_sum + p*q,
                    chosen + [q]
                )
        
        for Q in recurse(0, 0, []):
            solutions.append(list(zip(P, Q)))

    print(solutions)

single_step_ep(3,2,6,2)

(1, 1)
[[1, 2, 3], [1, 2, 3]]
(1, 2)
[[1, 2, 3], [1, 2, 3, 4]]
(1, 3)
[[1, 2, 3], [1, 2, 3]]
(1, 4)
[[1, 2, 3], [1, 2, 3, 4]]
(1, 5)
[[1, 2, 3], [1, 2, 3, 5]]
(2, 2)
[[1, 2, 3, 4], [1, 2, 3, 4]]
(2, 3)
[[1, 2, 3, 4], [1, 2, 3]]
(2, 4)
[[1, 2, 3, 4], [1, 2, 3, 4]]
(3, 3)
[[1, 2, 3], [1, 2, 3]]
[[(1, 3), (1, 3)], [(1, 2), (2, 2)], [(1, 3), (3, 1)], [(1, 2), (4, 1)], [(1, 1), (5, 1)], [(2, 1), (2, 2)], [(2, 2), (2, 1)], [(2, 1), (4, 1)], [(3, 1), (3, 1)]]
